## 네이버 금융 종목 뉴스 크롤러

의존성은 uv가 관리한다. `!pip install`은 잠긴 버전과 어긋날 수 있으므로 쓰지 않는다.

```bash
uv sync             # 의존성 설치 (requests, beautifulsoup4, ipykernel)
uv run jupyter lab  # 이 노트북 실행 (커널은 프로젝트 .venv 선택)
```

수집이 끝나면 `uv run python clean.py`로 정제한다.

### 수집 규모

종목 10개 × 목록 25페이지 × 최대 60건. 요청 간격 1초 기준 **20~30분** 소요된다.

중간에 끊겨도 `crawl_cache/`에 종목별로 저장돼 있어 다시 실행하면 이어서 진행한다.
처음부터 다시 받으려면 `crawl_cache/` 폴더를 지우면 된다.

In [ ]:
# -*- coding: utf-8 -*-
"""
네이버 금융 종목 뉴스 크롤러
- 대상 종목의 뉴스 목록 -> 개별 기사 본문 수집 -> JSON 저장
- 1주차 할일 2번: 뉴스 200~300건 수집

재수집 시 확대 사항
- pages 5 -> 25 : 1차 수집이 8/4~8/7 4일치에 그쳐 기간을 넓힌다
- 종목당 30 -> 60건 : 정제 단계에서 중복이 39% 걸러지므로 여유분이 필요하다
- 종목별 중간 저장 : 20~30분짜리 작업이라 중간에 끊겨도 이어받게 한다
- 요청 실패 재시도 : 요청량이 5배로 늘어 타임아웃/일시 차단 가능성이 커진다
"""

import requests
from bs4 import BeautifulSoup
import re
import time
import json
import os

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Referer": "https://finance.naver.com/item/main.naver",
}
REQUEST_DELAY = 1.0  # 서버 부담을 줄이기 위한 요청 간격(초)
LIST_PAGES = 25  # 종목당 목록 페이지 수 (뒤로 갈수록 과거 기사)
ARTICLES_PER_STOCK = 60  # 종목당 목표 수집 건수
MIN_CONTENT_LENGTH = 100  # 이보다 짧으면 단신으로 간주하고 제외
MAX_RETRY = 3  # 요청 실패 시 재시도 횟수
CACHE_DIR = "crawl_cache"  # 종목별 중간 저장 위치

# 확정된 10개 종목 (코스피200 상위 섹터 중복 제외)
STOCKS = {
    "005930": "삼성전자",        # 반도체(메모리)
    "000660": "SK하이닉스",      # 반도체(메모리)
    "005380": "현대차",          # 자동차
    "373220": "LG에너지솔루션",  # 2차전지
    "207940": "삼성바이오로직스", # 바이오/CDMO
    "105560": "KB금융",          # 금융(은행)
    "329180": "HD현대중공업",    # 조선
    "012450": "한화에어로스페이스", # 방산
    "034020": "두산에너빌리티",   # 원전/발전
    "035420": "NAVER",           # 인터넷 플랫폼
}

# 같은 기사가 여러 종목에 걸쳐 나온다(1차 수집 기준 39%).
# URL -> 본문 캐시를 두면 그만큼 요청을 줄일 수 있다.
_content_cache = {}


def clean_text(text: str) -> str:
    """공백/개행 정리"""
    return re.sub(r"\s+", " ", text).strip()


def fetch(url: str, encoding: str = "euc-kr"):
    """재시도가 붙은 GET. 끝내 실패하면 None을 돌려주고 호출 측에서 건너뛴다."""
    for attempt in range(1, MAX_RETRY + 1):
        try:
            res = requests.get(url, headers=HEADERS, timeout=15)
            # 429/503은 잠시 뒤 풀리는 경우가 많아 기다렸다 다시 시도
            if res.status_code in (429, 503):
                wait = REQUEST_DELAY * (2 ** attempt)
                print(f"    {res.status_code} 응답, {wait:.0f}초 대기 후 재시도")
                time.sleep(wait)
                continue
            res.raise_for_status()
            res.encoding = encoding
            return res
        except requests.RequestException as e:
            if attempt == MAX_RETRY:
                print(f"    요청 실패({type(e).__name__}), 건너뜀: {url[:70]}")
                return None
            time.sleep(REQUEST_DELAY * (2 ** attempt))
    return None


def get_news_list(code: str, pages: int = LIST_PAGES):
    """종목 뉴스 목록 페이지에서 (제목, 링크, 날짜) 수집"""
    links = []
    seen = set()  # 페이지 사이에 같은 기사가 반복되므로 URL 기준으로 걸러낸다
    for page in range(1, pages + 1):
        url = f"https://finance.naver.com/item/news_news.naver?code={code}&page={page}"
        res = fetch(url)
        if res is None:
            continue
        soup = BeautifulSoup(res.text, "html.parser")

        # 뉴스 목록은 table.type5 안에 tr로 구성됨 (구조 바뀌면 여기부터 확인)
        rows = soup.select("table.type5 tbody tr")
        if not rows:
            break  # 마지막 페이지를 지나면 빈 목록이 온다

        for row in rows:
            title_tag = row.select_one("td.title a")
            date_tag = row.select_one("td.date")
            if not title_tag:
                continue

            href = title_tag.get("href", "")
            full_url = "https://finance.naver.com" + href if href.startswith("/") else href
            if not full_url or full_url in seen:
                continue
            seen.add(full_url)

            links.append({
                "title": clean_text(title_tag.get_text()),
                "url": full_url,
                "date": clean_text(date_tag.get_text()) if date_tag else None,
            })

        time.sleep(REQUEST_DELAY)
    return links


def get_article_content(url: str):
    """개별 기사 페이지에서 본문 텍스트 추출"""
    if url in _content_cache:  # 다른 종목에서 이미 받아둔 기사
        return _content_cache[url]

    res = fetch(url)
    if res is None:
        return None

    # news_read.naver는 본문 없이 JS로 n.news.naver.com으로 리다이렉트만 함
    redirect_match = re.search(r"location\.href\s*=\s*'([^']+)'", res.text)
    if redirect_match:
        res = fetch(redirect_match.group(1), encoding="utf-8")
        if res is None:
            return None

    soup = BeautifulSoup(res.text, "html.parser")

    # iframe으로 본문을 로드하는 구버전 케이스도 대비
    iframe = soup.select_one("iframe#news_read_iframe, iframe#mainFrame")
    if iframe and iframe.get("src"):
        iframe_url = iframe["src"]
        if iframe_url.startswith("/"):
            iframe_url = "https://finance.naver.com" + iframe_url
        res = fetch(iframe_url, encoding="utf-8")
        if res is None:
            return None
        soup = BeautifulSoup(res.text, "html.parser")

    # 본문 후보 셀렉터 (네이버뉴스 통합형 / 금융뉴스 자체 페이지)
    body = (
        soup.select_one("#dic_area")
        or soup.select_one("#articleBodyContents")
        or soup.select_one("#content")
    )
    if not body:
        _content_cache[url] = None
        return None

    for tag in body.select("script, style"):
        tag.decompose()

    content = clean_text(body.get_text())
    _content_cache[url] = content
    return content


def crawl_stock(code: str, name: str):
    """종목 1개를 수집한다. 이미 받아둔 결과가 있으면 그대로 재사용한다."""
    cache_path = os.path.join(CACHE_DIR, f"{code}.json")
    if os.path.exists(cache_path):
        with open(cache_path, encoding="utf-8") as f:
            done = json.load(f)
        for a in done:  # 캐시된 본문도 재사용 대상에 넣는다
            _content_cache.setdefault(a["url"], a["content"])
        print(f"[{name}] 이미 수집됨 -> {len(done)}건 (건너뜀)")
        return done

    print(f"[{name}] 뉴스 목록 수집 중...")
    news_list = get_news_list(code)
    print(f"  목록 {len(news_list)}건 확보, 본문 수집 시작")

    articles = []
    for item in news_list:
        if len(articles) >= ARTICLES_PER_STOCK:
            break
        if not item["url"]:
            continue

        cached = item["url"] in _content_cache
        content = get_article_content(item["url"])
        if not content or len(content) < MIN_CONTENT_LENGTH:
            continue

        articles.append({
            "title": item["title"],
            "content": content,
            "date": item["date"],
            "stock_name": name,
            "stock_code": code,
            "url": item["url"],
        })
        if not cached:  # 캐시에서 꺼낸 경우엔 요청이 없었으므로 쉴 필요도 없다
            time.sleep(REQUEST_DELAY)

    os.makedirs(CACHE_DIR, exist_ok=True)
    with open(cache_path, "w", encoding="utf-8") as f:
        json.dump(articles, f, ensure_ascii=False, indent=2)

    print(f"  -> {len(articles)}건 수집")
    return articles


def crawl():
    all_articles = []
    for code, name in STOCKS.items():
        all_articles.extend(crawl_stock(code, name))

    for i, a in enumerate(all_articles, 1):  # id는 마지막에 한 번에 매긴다
        a["id"] = i
    return all_articles


if __name__ == "__main__":
    articles = crawl()
    with open("news_data.json", "w", encoding="utf-8") as f:
        json.dump(articles, f, ensure_ascii=False, indent=2)
    print(f"\n총 {len(articles)}건 저장 완료 -> news_data.json")

    dates = sorted(a["date"][:10] for a in articles if a["date"])
    if dates:
        print(f"수집 기간: {dates[0]} ~ {dates[-1]}")
    print(f"고유 URL: {len(set(a['url'] for a in articles))}건")
    print("\n다음: uv run python clean.py")